In [ ]:
# Import Library, saya menggunakan beberapa library seperti kMeans dan silhouette_score
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# saya membaca data set terlebih dahulu
df = pd.read_excel("../dataset/online_retail_II.xlsx")

# kemudian saya ingin lihat isi dari dataset
df.head()

# mengecek Jumlah data
df.shape

# kemudian saya mengecek type data nya 
df.info()

# Statik data nya men-describe
df.describe()

# pengecekan Missing Value
df.isnull().sum()

# penghapusan Missing Value
# Karena Customer ID dipakai sebagai identitas pelanggan.
df = df.dropna(subset=["Customer ID"])
df = df.dropna(subset=["Description"])

# cek apakah masih missing value
df.isnull().sum()

# tahap selanjutnya
# Cek Data yang Duplikat
df.duplicated().sum()

#menghapus data duplicate
df = df.drop_duplicates()

# setelah itu cek lagi
df.duplicated().sum()

# lanjut cek data yang tidak valid
# Menghapus Data Tidak Valid
# Invoice seperti C536379 Huruf C artinya Cancelled
df = df[~df["Invoice"].astype(str).str.startswith("C")]
# Quantity negatif juga tidak digunakan.
df = df[df["Quantity"] > 0]
# Harga nol
df = df[df["Price"] > 0]

# Ubah Tipe Data
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Membuat Kolom Baru
df["TotalPrice"] = df["Quantity"] * df["Price"]

# tentukan tanggal terakhir.
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

# buat tabel RFM.
rfm = df.groupby("Customer ID").agg({
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
    "Invoice": "nunique",
    "TotalPrice": "sum"
})

# Ganti nama kolom.
rfm.columns = ["Recency","Frequency","Monetary"]

# lihat hasil
rfm.head()

# Kalau langsung clustering nanti Monetary mendominasi.
# Maka distandarisasi.
scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm)

# Menentukan Jumlah Cluster
# Menggunakan Elbow Method. 
# KMeans
sse=[]

for i in range(1,11):

    model = KMeans(
        n_clusters=i,
        random_state=42
    )

    model.fit(rfm_scaled)

    sse.append(model.inertia_)

# Menentukan Jumlah Cluster
plt.figure(figsize=(8,5))

plt.plot(range(1,11),sse,marker='o')

plt.xlabel("Jumlah Cluster")

plt.ylabel("SSE")

plt.title("Elbow Method")

plt.show()

# lanjut ke K-Means
kmeans = KMeans(
    n_clusters=3,
    random_state=42
)

rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)
# lihat hasil
rfm.head()

# tahap Evaluasi
score = silhouette_score(
    rfm_scaled,
    rfm["Cluster"]
)

print(score)

# Visualisasi
plt.figure(figsize=(10,6))

plt.scatter(
    rfm["Frequency"],
    rfm["Monetary"],
    c=rfm["Cluster"],
    cmap="viridis"
)

plt.xlabel("Frequency")

plt.ylabel("Monetary")

plt.title("Customer Segmentation")

plt.show()

# Analisis Cluster
rfm.groupby("Cluster").mean()
